# 09 — SHAP Explainability

## Goal
Explain the model: global feature importance, local waterfall explanations, fairness audit and risk segmentation.

In [ ]:
from pathlib import Path
import sys

# Locate the project root (works whether the notebook is run from notebooks/ or root)
root = Path.cwd()
while not (root / "src" / "data_loader.py").exists() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root))
print("project root:", root)


In [ ]:
from pathlib import Path

missing = [p.name for p in [
    root / "models" / "metadata.json",
    root / "reports" / "model_results.csv",
] if not p.exists()]
if missing:
    print("Missing artifacts:", missing)
    print("Run first:  python -m src.train && python -m src.evaluate && python -m src.explain")
else:
    print("All artifacts present.")


In [ ]:
import joblib, pandas as pd, shap

xgb_pipe = joblib.load(root / "models" / "xgboost.joblib")
xgb = xgb_pipe.named_steps["clf"]
data = prepare_matrices()
X_sample = data["X_test"].sample(1500, random_state=42)
explainer = shap.TreeExplainer(xgb)
explanation = explainer(X_sample)
shap.plots.bar(explanation, max_display=12, show=False)


In [ ]:
shap.plots.beeswarm(explanation, max_display=12, show=False)


In [ ]:
import pickle
bundle = pickle.load(open(root / "reports" / "shap_bundle.pkl", "rb"))
hi = bundle["local"]["high_risk"]
print(f"High-risk customer: calibrated probability {hi['calibrated_probability']:.1%}, actual outcome = default")
pd.DataFrame(hi["top_drivers"])


In [ ]:
pd.read_csv(root / "reports" / "risk_segments.csv")


**Risk segmentation story** — Low band (<20%): 66% of clients, 10.9% observed default. Medium (20–50%): 23% of clients, 31.8%. High (≥50%): 11% of clients, 69.7% observed default — each band captures roughly a third of all actual defaulters. Fairness note: observed group-level differences (e.g. TPR by gender) do not by themselves establish discriminatory impact in a real lending institution.